In [1]:
!pip install sentencepiece protobuf tiktoken --upgrade

In [2]:
!pip install transformers accelerate bitsandbytes spacy seqeval pandas peft scikit-learn -q
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 151.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import os
import argparse
import torch
from torch.cuda.amp import GradScaler
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from config import (
    MODEL_NAME, BATCH_SIZE, EPOCHS, MAX_LEN, device,
    LABEL_MAP, ASPECT_LABEL_MAP, SENTIMENT_LABEL_MAP, ACCUM_STEPS, HF_TOKEN,
)
from data import parse_data, split_data, ABSADataset, UnlabeledDataset
from model import LLMSynABSA
from train import train_one_epoch, build_optimizers
from evaluate import evaluate
from utils import syntax_caching

In [4]:
class Args:
    source = "restaurant"
    target = "laptop"
    data_dir = "data"
    output_dir = "checkpoints"
    split_ratio = 0.8
    seed = 42

args = Args()
os.makedirs(args.output_dir, exist_ok=True)

In [5]:
print(f"Loading tokenizer: {MODEL_NAME}")
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading tokenizer: meta-llama/Meta-Llama-3-8B-Instruct


In [ ]:
# Paths
src_path = os.path.join(args.data_dir, f"{args.source}.csv")
tgt_path = os.path.join(args.data_dir, f"{args.target}.csv")

# Source domain
print(f"\nLoading source domain: {src_path}")
src_data = parse_data(src_path)
src_train, src_test = split_data(src_data, train_ratio=args.split_ratio, seed=args.seed)
print(f"  Total: {len(src_data)} | Train: {len(src_train)} | Test: {len(src_test)}")

# Target domain
print(f"Loading target domain: {tgt_path}")
tgt_data = parse_data(tgt_path)
tgt_train, tgt_test = split_data(tgt_data, train_ratio=args.split_ratio, seed=args.seed)
print(f"  Total: {len(tgt_data)} | Train(unlabeled): {len(tgt_train)} | Test: {len(tgt_test)}")


Loading source domain: data/restaurant.csv
  Total: 3497 | Train: 2797 | Test: 700
Loading target domain: data/laptop.csv
  Total: 1859 | Train(unlabeled): 1487 | Test: 372


In [7]:
# Datasets
src_dataset = ABSADataset(src_train, tokenizer, LABEL_MAP, ASPECT_LABEL_MAP, SENTIMENT_LABEL_MAP)
src_test_dataset = ABSADataset(src_test, tokenizer, LABEL_MAP, ASPECT_LABEL_MAP, SENTIMENT_LABEL_MAP)
tgt_train_dataset = UnlabeledDataset(tgt_train, tokenizer)
tgt_test_dataset = ABSADataset(tgt_test, tokenizer, LABEL_MAP, ASPECT_LABEL_MAP, SENTIMENT_LABEL_MAP)

# DataLoaders
src_loader = DataLoader(src_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
tgt_train_loader = DataLoader(tgt_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
tgt_test_loader = DataLoader(tgt_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [8]:
print("\nBuilding syntax cache...")
all_data = src_train + src_test + tgt_train + tgt_test
syntax_cache = syntax_caching(all_data, [], tokenizer)


Building syntax cache...
Building syntax cache for 5356 texts...
  500/5356
  1000/5356
  1500/5356
  2000/5356


  2500/5356
  3000/5356
  3500/5356
  4000/5356
  4500/5356
  5000/5356
Syntax cache built.


In [9]:
print(f"\nInitializing model...")
model = LLMSynABSA(MODEL_NAME).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  Total params:     {total:,}")
print(f"  Trainable params: {trainable:,}")
print(f"  Frozen params:    {total - trainable:,}")


Initializing model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

: 

In [10]:
opt_main, opt_syntax = build_optimizers(model)
scaler = GradScaler()

opt_main.zero_grad()
opt_syntax.zero_grad()

/tmp/ipykernel_241728/3673528720.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
best_f1 = 0.0
best_epoch = 0
save_path = None

print(f"\n{'='*60}")
print(f"Training: {args.source} → {args.target} ({EPOCHS} epochs)")
print(f"{'='*60}\n")


from torch.optim.lr_scheduler import LambdaLR  # <-- this line is needed

opt_main, opt_syntax = build_optimizers(model)

total_steps = EPOCHS * len(src_loader) // ACCUM_STEPS
warmup_steps = int(0.1 * total_steps)

def lr_lambda(step):
    # warmup
    if step < warmup_steps:
        return step / warmup_steps

    # linear decay with floor
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)

    return max(0.01, 1.0 - progress)

scheduler_main = LambdaLR(
    opt_main,
    lr_lambda=lr_lambda
)

best_f1 = 0.0
patience = 10
patience_counter = 0

# training loop
for epoch in range(1, EPOCHS+1):
    avg_loss, avg_seq, avg_adv = train_one_epoch(
        model, src_loader, tgt_train_loader,
        (opt_main, opt_syntax), scheduler_main,
        syntax_cache, epoch, EPOCHS
    )

    if epoch % 1 == 0 or epoch == 1:

        results = evaluate(model, tgt_test_loader, tokenizer, syntax_cache)

        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"Loss: {avg_loss:.4f} (seq: {avg_seq:.4f} adv: {avg_adv:.4f}) | "
            f"Macro-F1: {results['macro_f1']:.4f} | "
            f"Micro-F1: {results['micro_f1']:.4f} | "
            f"P: {results['precision']:.4f} R: {results['recall']:.4f}"
        )

        if results["macro_f1"] > best_f1:

            best_f1 = results["macro_f1"]
            best_epoch = epoch
            patience_counter = 0

            save_path = os.path.join(
                args.output_dir,
                f"{args.source}_to_{args.target}_best.pt"
            )

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "macro_f1": best_f1,
                "results": results,
            }, save_path)

            print(f"  >> New best! Saved to {save_path}")

        else:
            patience_counter += 1

    else:

        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"Loss: {avg_loss:.4f} "
            f"(seq: {avg_seq:.4f} adv: {avg_adv:.4f})"
        )


Training: restaurant → laptop (100 epochs)



Epoch   1/100 | Loss: 1.2931 (seq: 1.2915 adv: 0.4862) | Macro-F1: 0.0772 | Micro-F1: 0.1050 | P: 0.1213 R: 0.0925
  >> New best! Saved to checkpoints/restaurant_to_laptop_best.pt
Epoch   2/100 | Loss: 0.7030 (seq: 0.7021 adv: 0.1393) | Macro-F1: 0.0688 | Micro-F1: 0.0789 | P: 0.1310 R: 0.0565
Epoch   3/100 | Loss: 0.5470 (seq: 0.5466 adv: 0.0419) | Macro-F1: 0.0989 | Micro-F1: 0.1271 | P: 0.1667 R: 0.1027
  >> New best! Saved to checkpoints/restaurant_to_laptop_best.pt
Epoch   4/100 | Loss: 0.4580 (seq: 0.4578 adv: 0.0157) | Macro-F1: 0.1368 | Micro-F1: 0.1793 | P: 0.2427 R: 0.1421
  >> New best! Saved to checkpoints/restaurant_to_laptop_best.pt
Epoch   5/100 | Loss: 0.3552 (seq: 0.3551 adv: 0.0088) | Macro-F1: 0.2209 | Micro-F1: 0.2646 | P: 0.2163 R: 0.3408
  >> New best! Saved to checkpoints/restaurant_to_laptop_best.pt
Epoch   6/100 | Loss: 0.2427 (seq: 0.2426 adv: 0.0049) | Macro-F1: 0.2595 | Micro-F1: 0.2955 | P: 0.3305 R: 0.2671
  >> New best! Saved to checkpoints/restaurant_to_

In [10]:
final_path = os.path.join(args.output_dir, f"{args.source}_to_{args.target}_final.pt")
torch.save({
    "epoch": EPOCHS,
    "model_state_dict": model.state_dict(),
    "macro_f1": best_f1,
}, final_path)

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best Macro-F1: {best_f1:.4f} at epoch {best_epoch}")
print(f"Final model: {final_path}")
print(f"Best model:  {save_path if save_path else 'N/A'}")
print(f"{'='*60}")

NameError: name 'model' is not defined